# YOLO 画像分類サンプル

## YOLO 画像分類

このノートブックは **Ultralytics YOLO** を使った画像分類のサンプルです。

```
画像（URL またはファイル）→ YOLO 分類器 → 上位 N 件の予測ラベル + 信頼スコア
```

検出（複数物体へのバウンディングボックス）とは異なり、分類は画像全体に対して**1 つのラベル**を割り当て、信頼スコアが高い順に上位クラスを返します。

### 使用可能なモデル

| モデル | パラメータ数 | 速度 |
|-------|------------|------|
| `yolo11x-cls.pt` | 56.9 M | 最も遅い / 最も高精度 |
| `yolo11l-cls.pt` | 25.3 M | — |
| `yolo11m-cls.pt` | 20.1 M | — |
| `yolo11s-cls.pt` |  9.4 M | — |
| `yolo11n-cls.pt` |  2.6 M | 最も速い / 最も軽量 |

writefile セル内の `MODEL_NAME` を変更することでモデルを切り替えられます。

📄 [Ultralytics YOLO ドキュメント](https://docs.ultralytics.com/)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/yolo"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls

In [ ]:
# Download sample images from GitHub
import os

# Image files to download
IMAGE_FILES = [
    'teefarm-pile-1651945_640.jpg',
]

BASE_URL = 'https://raw.githubusercontent.com/mastnk/cv-samples/main/yolo'
for fname in IMAGE_FILES:
    url = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Install Ultralytics
!pip install ultralytics -q

import ultralytics
ultralytics.checks()  # check environment

## 独自画像を使うには

画像を用意する方法は 2 つあります。

**① URL を指定する**  
スクリプト実行時に `--url` フラグに直接画像 URL を渡します：
```
%run classification.py --url https://example.com/image.jpg
```

**② 画像ファイルを `PROJECT_PATH/` に置く**  
画像ファイルを `PROJECT_PATH/` に配置してから `--file` または `--dir` で実行します。

アップロードは **Google Drive** 経由が簡単です：  
[drive.google.com](https://drive.google.com) を開き、`マイドライブ / CV-Samples / yolo/` に移動してファイルをドラッグ＆ドロップするだけ。  
マウント済みの Drive を通じて Colab から即座にアクセスできます。追加のアップロード手順は不要です。

```
%run classification.py --file your_image.jpg   # ファイル 1 枚
%run classification.py --dir  .                # フォルダ内の全画像
```

## モデルを選択するには

下の writefile セル内の `MODEL_NAME` を変更してモデルを切り替えます。  
`MODEL_NAME = ...` の行が複数ある場合、**有効になるのは最後の 1 行だけ**です。

```python
# MODEL_NAME = 'yolo11x-cls.pt'  # 56.9M params — most accurate, slowest
# MODEL_NAME = 'yolo11l-cls.pt'  # 25.3M params
# MODEL_NAME = 'yolo11m-cls.pt'  # 20.1M params
# MODEL_NAME = 'yolo11s-cls.pt'  #  9.4M params
MODEL_NAME   = 'yolo11n-cls.pt'  #  2.6M params — fastest, lightest  ← active
```

モデルが大きいほど精度は高くなりますが、実行時間も長くなります。まずは `yolo11n-cls.pt` で試してみましょう。

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile classification.py
"""YOLO Image Classification — command-line interface.

Usage:
  %run classification.py --url  <image_url>  [--disp] [--topk INT]
  %run classification.py --file <image_path> [--disp] [--topk INT]
  %run classification.py --dir  <image_dir>  [--disp] [--topk INT]
"""

import argparse
import glob
import os

from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME   = 'yolo11n-cls.pt'  #  2.6M params — fastest / lightest
# MODEL_NAME = 'yolo11s-cls.pt'  #  9.4M params
# MODEL_NAME = 'yolo11m-cls.pt'  # 20.1M params
# MODEL_NAME = 'yolo11l-cls.pt'  # 25.3M params
# MODEL_NAME = 'yolo11x-cls.pt'  # 56.9M params — most accurate

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/yolo'

# ---- Model loading -----------------------------------------------
model = YOLO(MODEL_NAME)

# ---- Display helper ----------------------------------------------
def show(image: Image.Image, label: str, disp: bool) -> None:
    """When --disp is active, display the image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(image)
        print(label)

# ---- Functions ---------------------------------------------------
def classify_url(url: str, topk: int = 5, disp: bool = False):
    """Download an image from a URL and classify it."""
    image   = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    results = model.predict(image, verbose=False)
    show(image, url, disp)
    probs = results[0].probs
    names = results[0].names
    top_indices = probs.top5[:topk]
    top_confs   = probs.top5conf[:topk].tolist()
    for idx, conf in zip(top_indices, top_confs):
        print(f'  {names[idx]:<30} {conf*100:.1f}%')
    return results


def classify_file(path: str, topk: int = 5, disp: bool = False):
    """Classify a single local image file."""
    image   = Image.open(path).convert('RGB')
    results = model.predict(image, verbose=False)
    show(image, path, disp)
    probs = results[0].probs
    names = results[0].names
    top_indices = probs.top5[:topk]
    top_confs   = probs.top5conf[:topk].tolist()
    for idx, conf in zip(top_indices, top_confs):
        print(f'  {names[idx]:<30} {conf*100:.1f}%')
    return results


def classify_dir(directory: str, topk: int = 5, disp: bool = False):
    """Classify all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        classify_file(path, topk, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='YOLO Image Classification')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str, help='Image URL to classify')
group.add_argument('--file', type=str, help='Path to a single image file')
group.add_argument('--dir',  type=str, help='Directory of images to classify')
parser.add_argument('--topk', type=int, default=5,
                    help='Number of top predictions to show (default: 5)')
parser.add_argument('--disp', action='store_true',
                    help='Display image and filename/URL before results')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    classify_url(args.url,   topk=args.topk, disp=args.disp)
elif args.file:
    classify_file(args.file, topk=args.topk, disp=args.disp)
elif args.dir:
    classify_dir(args.dir,   topk=args.topk, disp=args.disp)


## `classification.py` の使い方

上の `%%writefile` セルを実行すると、`classification.py` が `PROJECT_PATH` に保存されます。  
`--disp` でのインライン画像表示を有効にするため、`!python` ではなく **`%run`** で実行してください。

```
%run classification.py --url  <画像URL>        # リモート画像の分類
%run classification.py --file <画像パス>       # ローカルファイル 1 枚の分類
%run classification.py --dir  <ディレクトリ>  # フォルダ内の全画像を分類
```

**オプション引数**

| フラグ | デフォルト | 説明 |
|--------|-----------|------|
| `--disp` | オフ | 結果の前に画像とファイル名 / URL を表示 |
| `--topk <n>` | `5` | 表示する上位予測数（1 〜 5） |

**実行例**

```bash
# リモート画像を分類して表示
%run classification.py --url https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_640.jpg --disp

# ファイル 1 枚、上位 1 件のみ表示
%run classification.py --file teefarm-pile-1651945_640.jpg --topk 1 --disp

# フォルダ内の全画像を分類（各画像を表示）
%run classification.py --dir . --disp
```

**出力形式（`--disp` あり）**

```
<画像をインライン表示>
teefarm-pile-1651945_640.jpg
  golden_retriever               82.4%
  Labrador_retriever             10.1%
  tennis_ball                     3.2%
  ...
```

## 実行方法

`classification.py` は `!python` ではなく **`%run`** で実行してください。Colab カーネル内で実行されるため、`--disp` でのインライン画像表示が有効になります。

| モード | フラグ | 説明 |
|--------|--------|------|
| URL から | `--url <url>` | リモート画像を取得して分類 |
| ファイル 1 枚 | `--file <パス>` | ローカル画像 1 枚を分類 |
| ディレクトリ | `--dir <パス>` | フォルダ内の全画像を分類 |

`--disp` を追加すると、結果の前に各画像とファイル名 / URL が表示されます。  
`--topk <n>` で表示する上位予測数を変更できます（デフォルト: 5）。

In [ ]:
# Execute: classify an image from URL (with display)
%cd "{PROJECT_PATH}"
%run classification.py --disp --url https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_640.jpg


In [ ]:
# Execute: classify a single local image file (with display)
%cd "{PROJECT_PATH}"
%run classification.py --disp --file teefarm-pile-1651945_640.jpg


In [ ]:
# Execute: classify all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run classification.py --disp --dir .


💾 **ノートブックを保存するのを忘れずに！**

作業内容を保持するため、閉じる前に **ファイル → ドライブにコピーを保存** を実行してください。